# Elementary Control Tables — Pipeline Observability

This notebook queries the **Elementary** monitoring tables (written by dbt-elementary into the `workspace.elementary` schema) to give a full picture of pipeline health:

| Section | What it shows |
|---|---|
| 1. Session Setup | Connect to Databricks |
| 2. Model Run History | Which models ran, how long, pass/fail |
| 3. Latest Run Status | Snapshot of most recent run per model |
| 4. Test Results | Which dbt tests passed / failed / warned |

---
## 1. Session Setup

In [40]:
from dotenv import load_dotenv
from databricks.connect import DatabricksSession
import os
import pandas as pd

load_dotenv()

spark = DatabricksSession.builder \
    .host(os.getenv("DATABRICKS_HOST")) \
    .token(os.getenv("DATABRICKS_TOKEN")) \
    .serverless(True) \
    .getOrCreate()
    
pd.set_option('display.max_colwidth', None)   # no truncation
pd.set_option('display.max_columns', None)    # show all columns
pd.set_option('display.max_rows', 50)

---
## 2. Model Run History

Elementary writes one row per model per `dbt run` invocation into `dbt_run_results`.  
This cell shows the **last 100 model executions** across all runs

In [11]:
spark.sql("""
    SELECT
        generated_at,                         
        unique_id,
        status,
        round(execution_time, 2)                AS execution_time,
        rows_affected,
        materialization,
        execute_started_at,
        execute_completed_at
    FROM workspace.elementary.dbt_run_results
    ORDER BY generated_at DESC
    LIMIT 100
""").toPandas()

,generated_at,unique_id,status,execution_time,rows_affected,materialization,execute_started_at,execute_completed_at
0,2026-04-26 15:26:03,model.elementary.dbt_invocations,success,27.870001,NaN,incremental,2026-04-26T15:24:16.120317Z,2026-04-26T15:24:43.737915Z
1,2026-04-26 15:26:03,model.ecommerce_data_engineering.dim_products,success,4.980000,NaN,table,2026-04-26T15:25:25.541969Z,2026-04-26T15:25:30.455196Z
2,2026-04-26 15:26:03,test.ecommerce_data_engineering.not_null_fct_order_items_product_id.6fbedef7dc,pass,2.470000,NaN,test,2026-04-26T15:25:44.079578Z,2026-04-26T15:25:46.429956Z
3,2026-04-26 15:26:03,test.ecommerce_data_engineering.not_null_stg_orders_order_date.2177a3e8bb,pass,2.900000,NaN,test,2026-04-26T15:25:13.913052Z,2026-04-26T15:25:16.745678Z
4,2026-04-26 15:26:03,test.ecommerce_data_engineering.relationships_fct_orders_customer_id__customer_id__ref_scd_customers_.3306ff43bb,pass,2.800000,NaN,test,2026-04-26T15:25:56.052215Z,2026-04-26T15:25:58.794094Z
...,...,...,...,...,...,...,...,...
95,2026-04-26 15:19:22,test.ecommerce_data_engineering.not_null_stg_orders_customer_id.af79d5e4b5,pass,4.080000,NaN,test,2026-04-26T15:18:34.014668Z,2026-04-26T15:18:36.851353Z
96,2026-04-26 15:19:22,test.ecommerce_data_engineering.not_null_stg_order_items_product_id.5154a8a6f6,pass,3.870000,NaN,test,2026-04-26T15:18:29.825139Z,2026-04-26T15:18:32.761421Z
97,2026-04-26 15:19:22,model.elementary.dbt_snapshots,success,46.810001,NaN,incremental,2026-04-26T15:16:58.185299Z,2026-04-26T15:17:44.678812Z
98,2026-04-26 15:19:22,model.elementary.dbt_artifacts_hashes,success,3.780000,NaN,view,2026-04-26T15:18:05.836425Z,2026-04-26T15:18:08.655803Z


---
## 3. Latest Run Status per Model

One row per model, showing only the **most recent** execution.  
A quick health check — everything should be `success`.

In [33]:
spark.sql("""
    WITH ranked AS (
        SELECT
            invocation_id,
            unique_id,
            materialization,
            status,
            round(execution_time, 2) AS execution_time_sec,
            rows_affected,
            generated_at,
            ROW_NUMBER() OVER (
                PARTITION BY unique_id
                ORDER BY generated_at DESC
            ) AS rn
        FROM workspace.elementary.dbt_run_results
        WHERE unique_id NOT LIKE '%elementary%'
        and unique_id not like '%test%'
    )
    SELECT
        invocation_id,
        unique_id,
        materialization,
        status,
        execution_time_sec,
        rows_affected,
        generated_at
    FROM ranked
    WHERE rn = 1
""").toPandas()

,invocation_id,unique_id,materialization,status,execution_time_sec,rows_affected,generated_at
0,00bf9b65-d7ba-406a-8580-b5977d46557d,model.ecommerce_data_engineering.dim_customers,table,success,4.760000,NaN,2026-04-27 03:34:23
1,00bf9b65-d7ba-406a-8580-b5977d46557d,model.ecommerce_data_engineering.dim_products,table,success,5.800000,NaN,2026-04-27 03:34:23
2,00bf9b65-d7ba-406a-8580-b5977d46557d,model.ecommerce_data_engineering.fct_order_items,incremental,success,18.379999,NaN,2026-04-27 03:34:23
3,00bf9b65-d7ba-406a-8580-b5977d46557d,model.ecommerce_data_engineering.fct_orders,incremental,success,14.290000,NaN,2026-04-27 03:34:23
4,00bf9b65-d7ba-406a-8580-b5977d46557d,model.ecommerce_data_engineering.int_order_items_with_product,view,success,2.950000,NaN,2026-04-27 03:34:23
5,00bf9b65-d7ba-406a-8580-b5977d46557d,model.ecommerce_data_engineering.int_orders_enriched,view,success,1.870000,NaN,2026-04-27 03:34:23
6,00bf9b65-d7ba-406a-8580-b5977d46557d,model.ecommerce_data_engineering.stg_customers,view,success,2.390000,NaN,2026-04-27 03:34:22
7,00bf9b65-d7ba-406a-8580-b5977d46557d,model.ecommerce_data_engineering.stg_order_items,view,success,2.950000,NaN,2026-04-27 03:34:22
8,00bf9b65-d7ba-406a-8580-b5977d46557d,model.ecommerce_data_engineering.stg_orders,view,success,3.040000,NaN,2026-04-27 03:34:22
9,00bf9b65-d7ba-406a-8580-b5977d46557d,model.ecommerce_data_engineering.stg_products,view,success,2.490000,NaN,2026-04-27 03:34:22


---
## 4. Test Results Summary

Elementary captures every `dbt test` execution in `dbt_run_results`.  
This summary groups by test status so you can see how many tests passed, failed, or warned at a glance.

In [42]:

spark.sql("""
    WITH ranked AS (
        SELECT
            a.unique_id,
            b.type,
            a.status,
            round(execution_time, 2) AS execution_time_sec,
            a.generated_at,
            ROW_NUMBER() OVER (
                PARTITION BY a.unique_id
                ORDER BY a.generated_at DESC
            ) AS rn
        FROM workspace.elementary.dbt_run_results a
        left join workspace.elementary.dbt_tests b
            on a.unique_id = b.unique_id
        WHERE a.unique_id like 'test%'
    )
    SELECT
        *
    FROM ranked
    WHERE rn = 1
""").toPandas()


,unique_id,type,status,execution_time_sec,generated_at,rn
0,test.ecommerce_data_engineering.accepted_values_fct_orders_status__delivered__shipped__processing__returned__cancelled.a06f456b86,generic,pass,3.56,2026-04-27 03:34:23,1
1,test.ecommerce_data_engineering.accepted_values_stg_orders_status__delivered__shipped__processing__returned__cancelled.901988bb47,generic,pass,4.57,2026-04-27 03:34:23,1
2,test.ecommerce_data_engineering.assert_customer_has_valid_email,singular,pass,1.38,2026-04-27 03:34:23,1
3,test.ecommerce_data_engineering.assert_fct_orders_revenue_non_negative,singular,pass,2.61,2026-04-27 03:34:23,1
4,test.ecommerce_data_engineering.assert_no_future_order_dates,singular,pass,3.80,2026-04-27 03:34:23,1
5,test.ecommerce_data_engineering.assert_product_price_positive,singular,pass,4.59,2026-04-27 03:34:23,1
6,test.ecommerce_data_engineering.assert_quantity_positive,singular,pass,3.23,2026-04-27 03:34:23,1
7,test.ecommerce_data_engineering.not_null_dim_customers_customer_id.dd91cd1c8d,generic,pass,2.49,2026-04-27 03:34:23,1
8,test.ecommerce_data_engineering.not_null_dim_products_category.b4c6ea8fb0,generic,pass,2.25,2026-04-27 03:34:23,1
9,test.ecommerce_data_engineering.not_null_dim_products_product_id.c8aba288d1,generic,pass,1.97,2026-04-27 03:34:23,1
